In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, make_scorer
from scipy.stats import randint, uniform
import joblib
from datetime import datetime
import multiprocessing
from scipy.stats import randint, uniform, loguniform



In [16]:
pd.set_option('display.max_columns', None)  # mostra tutte le colonne
pd.set_option('display.width', None)        # evita l'andata a capo automatica

In [17]:
def evaluate_model(y_true, y_pred, verbose=True):
    """
    Calculates multiple regression metrics and returns them as a DataFrame.
    
    Parameters:
    - y_true: array-like, true values
    - y_pred: array-like, predicted values
    - verbose: bool, if True prints the results
    
    Returns:
    - df_metrics: pd.DataFrame with all metrics
    """
    # RMSE
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    # SMAPE
    smape = (100/len(y_true)) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))
    
    metrics = {
        # MSE (Mean Squared Error), 0 -> Inf, più basso è meglio
        "MSE": mean_squared_error(y_true, y_pred),
        
        # RMSE (Root Mean Squared Error), 0 -> Inf, più basso è meglio ma stessa unità della variabile target, più interpretabile
        "RMSE": rmse,
        
        # # RMSE% (Root Mean Squared Error percentage) 0 -> Inf e maggiore è il RMSE%, peggiore è la previsione.
        "RMSE%": (rmse / np.mean(y_true)) * 100,
        
        # MAE (Mean Absolute Error), 0 -> inf, meno sensibile ai valori estremi rispetto a MSE
        "MAE": mean_absolute_error(y_true, y_pred),
        
        # R^2 (Coefficient of Determination), -Inf -> 1, più vicino a 1 meglio
        "R^2": r2_score(y_true, y_pred),
        
        # MAPE (Mean Absolute Percentage Error) con range 0 -> Inf, esprime errore in percentuale. Maggiore è il MAPE, peggiore è la previsione.
        "MAPE%": (abs(y_true - y_pred) / y_true).mean() * 100,

        # SMAPE (Symmetric Mean Absolute Percentage Error) con range 0 -> Inf, più robusto del MAPE per valori vicini a zero
        "SMAPE": smape
    }
    
    df_metrics = pd.DataFrame.from_dict(metrics, orient="index", columns=["Value"])
    
    if verbose:
        print(df_metrics)
    
    return df_metrics

### Start Analisys

In [18]:
df = pd.read_csv("tot_PIE1_final - Copia.csv", sep=';', parse_dates=["ts"])

# assicurati che sia ordinato temporalmente
df = df.sort_values("ts") 

# Mostra le prime righe del dataset
print(df.head(3))

  Site_code species_clean                ts  doy  year  day  month  hour  \
0      PIE1    fsylvatica  01/01/2018 00:00    1  2018    1      1     0   
1      PIE1    fsylvatica  01/01/2018 01:00    1  2018    1      1     1   
2      PIE1    fsylvatica  01/01/2018 02:00    1  2018    1      1     2   

   week  Rain_tot  AirTemp_Avg  RH_Avg  AirPres_Avg  Rg_Avg  WindSpeed  \
0     1       0.0          2.6    94.6          NaN     0.0      0.278   
1     1       0.0          2.6    94.8          NaN     0.0      0.274   
2     1       0.0          2.4    95.3          NaN     0.0      0.569   

   WindDir  O3_Avg        ozone       ozonec  ozoneppb  aot40h  sum60h  pod0  \
0  278.100   33.27  1483.287500  1877.571169     33.27     0.0     0.0   0.0   
1  314.100   32.68  1456.983333  1849.715030     32.68     0.0     0.0   0.0   
2    0.922   33.11  1476.154167  1671.592003     33.11     0.0     0.0   0.0   

   SoilTemp_Avg   VWC_Avg  LMA  gmax       vpd  fvpd  Dendro_1  Dendro_2  \
0

In [19]:
df = df.drop(["Site_code", "species_clean", "ts", "AirPres_Avg", "Dendro_1", "Dendro_2", "Dendro_3", "Dendro_4"], axis=1)
print(df.head(3))

   doy  year  day  month  hour  week  Rain_tot  AirTemp_Avg  RH_Avg  Rg_Avg  \
0    1  2018    1      1     0     1       0.0          2.6    94.6     0.0   
1    1  2018    1      1     1     1       0.0          2.6    94.8     0.0   
2    1  2018    1      1     2     1       0.0          2.4    95.3     0.0   

   WindSpeed  WindDir  O3_Avg        ozone       ozonec  ozoneppb  aot40h  \
0      0.278  278.100   33.27  1483.287500  1877.571169     33.27     0.0   
1      0.274  314.100   32.68  1456.983333  1849.715030     32.68     0.0   
2      0.569    0.922   33.11  1476.154167  1671.592003     33.11     0.0   

   sum60h  pod0  SoilTemp_Avg   VWC_Avg  LMA  gmax       vpd  fvpd  pod1  \
0     0.0   0.0      2.210000  0.363000   61    69  0.039772   1.0   0.0   
1     0.0   0.0      2.209797  0.363627   61    69  0.038299   1.0   0.0   
2     0.0   0.0      2.206022  0.364515   61    69  0.034127   1.0   0.0   

   Dendro_mean  
0     1.211307  
1     1.211307  
2     1.211307  


In [20]:
print(f"Size dataset: {len(df)}")

FEATURES = ["doy", "year", "day", "month", "hour", "week", "Rain_tot", "AirTemp_Avg", "RH_Avg", 
            "Rg_Avg", "WindSpeed", "WindDir", "O3_Avg", "ozone", "ozonec", "ozoneppb", 
           "aot40h", "sum60h", "pod0", "SoilTemp_Avg", "VWC_Avg", "LMA", "gmax", "vpd", "fvpd", "pod1"]
TARGET = "Dendro_mean"

X = df[FEATURES]
y = df[TARGET]

# set train size
# train_size = int(len(df) * 0.8)
train_size = int(len(df) * 0.9)

# Split temporale
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"Size train {len(X_train)} and size test {len(X_test)}")

display(X_train.head(3))
display(X_train.tail(3))
display(X_test.head(3))

Size dataset: 59904
Size train 53913 and size test 5991


,doy,year,day,month,hour,week,Rain_tot,AirTemp_Avg,RH_Avg,Rg_Avg,WindSpeed,WindDir,O3_Avg,ozone,ozonec,ozoneppb,aot40h,sum60h,pod0,SoilTemp_Avg,VWC_Avg,LMA,gmax,vpd,fvpd,pod1
0,1,2018,1,1,0,1,0.0,2.6,94.6,0.0,0.278,278.100,33.27,1483.287500,1877.571169,33.27,0.0,0.0,0.0,2.210000,0.363000,61,69,0.039772,1.0,0.0
1,1,2018,1,1,1,1,0.0,2.6,94.8,0.0,0.274,314.100,32.68,1456.983333,1849.715030,32.68,0.0,0.0,0.0,2.209797,0.363627,61,69,0.038299,1.0,0.0
2,1,2018,1,1,2,1,0.0,2.4,95.3,0.0,0.569,0.922,33.11,1476.154167,1671.592003,33.11,0.0,0.0,0.0,2.206022,0.364515,61,69,0.034127,1.0,0.0


,doy,year,day,month,hour,week,Rain_tot,AirTemp_Avg,RH_Avg,Rg_Avg,WindSpeed,WindDir,O3_Avg,ozone,ozonec,ozoneppb,aot40h,sum60h,pod0,SoilTemp_Avg,VWC_Avg,LMA,gmax,vpd,fvpd,pod1
38598,148,2022,28,5,6,22,0.0,16.87,62.14,42.28,0.606,351.1,46.04,2052.616667,2308.073255,46.04,0.00,0.0,1.735746,12.14524,0.349100,61,69,0.727602,1.000000,0.735746
38599,148,2022,28,5,7,22,0.0,18.10,59.42,38.02,0.637,118.7,41.90,1868.041667,2089.401133,41.90,0.00,0.0,1.404560,12.10338,0.348505,61,69,0.842848,0.967403,0.404560
38600,148,2022,28,5,8,22,0.0,18.78,63.53,50.89,1.699,141.2,46.68,2081.150000,2174.582181,46.68,6.68,0.0,1.870131,12.06012,0.348420,61,69,0.790456,1.000000,0.870131


,doy,year,day,month,hour,week,Rain_tot,AirTemp_Avg,RH_Avg,Rg_Avg,WindSpeed,WindDir,O3_Avg,ozone,ozonec,ozoneppb,aot40h,sum60h,pod0,SoilTemp_Avg,VWC_Avg,LMA,gmax,vpd,fvpd,pod1
38601,148,2022,28,5,9,22,0.0,19.40,60.14,411.9,2.885,143.7,49.57,2209.995833,2268.576970,49.57,9.57,0.0,3.232021,12.02655,0.347730,61,69,0.897978,0.925461,2.232021
38602,148,2022,28,5,10,22,0.0,19.67,67.66,677.9,3.180,147.6,48.73,2172.545833,2224.809972,48.73,8.73,0.0,3.411263,12.01411,0.346975,61,69,0.740890,1.000000,2.411263
38603,148,2022,28,5,11,22,0.0,19.76,70.06,748.8,3.461,154.4,50.46,2249.675000,2299.414240,50.46,10.46,0.0,3.527892,12.02418,0.346075,61,69,0.689748,1.000000,2.527892


In [21]:
# Setting XGB

# Controlla se XGBoost rileva CUDA
try:
    from xgboost import cuda
    gpu_available = True
except ImportError:
    gpu_available = False

USE_GPU = gpu_available # True se vuoi GPU, False se vuoi CPU
print(f"Is GPU available: {gpu_available}")

Is GPU available: False


## SETTING FOR ANDREA

In [22]:
### CHANGE PARAMS HERE
# for XGB
BOOSTER = "dart" # <- CHANGE HERE!!!!!!!!! (gbtree or dart). dart is slow but better then gbtree!!!
SET_TREE_METHOD = "exact" # <-  CHANGE HERE!!!!!!!!! (hist or exact). "exact" more precise but more slow of hist!!!

# for RandomizedSearchCV
N_ITER=2 # <- CHANGE HERE if training is too slow. try to change in 800, 500, 250 or 100!!!
SET_CV=5 # <- YOU CAN CHANGE in 3, but 5 is better!!!

In [23]:
# Setting for saving
MODEL = "xgb" # NOT CHANGE
OPTIMIZER = "randomizedSearch" # NOT CHANGE

In [24]:
# Definisci spazio iperparametri

# Definisci lo spazio degli iperparametri

# set GPU or CPU parameters
fixed_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "verbosity": 0,
    "booster": BOOSTER,
    "device": "cuda" if gpu_available else "cpu",
    "tree_method": "hist" if gpu_available else "exact", # "exact" (non supportato su GPU) per dataset piccolo ("hist" is valid also for CPU)
    "predictor": "gpu_predictor" if gpu_available else "cpu_predictor", # Usa la GPU o la CPU per le predizioni
    "sampling_method": "gradient_based" if gpu_available else "uniform", # "gradient_based" only for GPU
    "importance_type": "gain" if BOOSTER != "gblinear" else "weight",
    "n_jobs": 1 if gpu_available else multiprocessing.cpu_count(), # GPU=1 fai tutto il lavoro sulla GPU e CPU multiprocessing.cpu_count() usa tutti i core CPU
    "verbosity": 0
}

suggested_params = {
    "n_estimators": randint(500, 2000), # int, Numero di alberi
    "learning_rate": loguniform(0.01, 0.3), # float, Velocità di apprendimento (eta) [0.01, 0.3]
    "gamma": loguniform(0.01, 0.5), # float, Min miglioramento loss per effettuare uno split  (min_split_loss) [0.01, 0.5]
    "max_depth": randint(3, 15), # int, Profondità massima di ogni albero
    "min_child_weight": uniform(0.5, 9.5), # float, Min peso somma delle osservazioni per nodo foglia [0.5, 10]
    "max_delta_step": randint(0, 3), # int, Limite massimo per aggiornare il peso delle foglie
    "subsample":uniform(loc=0.5, scale=0.5), # float, Percentuale di campioni per ogni albero [0.5, 1.0]
    "colsample_bytree": uniform(loc=0.3, scale=0.7), # [0.3, 1.0]
    "colsample_bylevel": uniform(loc=0.3, scale=0.7), # [0.3, 1.0]
    "lambda": loguniform(1e-3, 2.0), # L2 regularization [0.001, 2.0]
    "alpha": loguniform(1e-3, 2.0), # L1 regularization (riduce overfitting) # [0.001, 2.0]
    "grow_policy": ["depthwise", "lossguide"], # Strategia di crescita degli alberi
}

tree_method = fixed_params["tree_method"]

if tree_method != "exact":
    suggested_params.update({"colsample_bynode": uniform(loc=0.3, scale=0.7)}) # [0.3, 1.0]

if BOOSTER == "dart":
    suggested_params.update({
        "sample_type": ["uniform", "weighted"],
        "normalize_type": ["tree", "forest"],
        "rate_drop": uniform(loc=0.2, scale=0.3), # 0.5 - 0.2 = 0.3 Probabilità di drop di un albero durante training [0.2, 0.5]
        "skip_drop": uniform(loc=0.1, scale=0.4),  # 0.4 = 0.5 - 0.1, Probabilità di saltare il drop in un'iterazione [0.1, 0.5]
        "one_drop": [True], # almeno un albero viene sempre droppato   
    })

In [ ]:
xgb = XGBRegressor(**fixed_params)



# Definisci TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=suggested_params,
    n_iter=N_ITER,
    scoring="neg_mean_squared_error",
    cv=tscv,
    verbose=2,
    n_jobs=1 # set to 1 to avoid xgb conflit 
)

# RandomizedSearchCV solo sul training
random_search.fit(X_train, y_train)

# get best model
best_model = random_search.best_estimator_

# Formatta timestamp: YYYYMMDDTHHMMSS
timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")

# save best model
joblib.dump(best_model, f"best_{MODEL}_{tree_method}_{OPTIMIZER}_{timestamp}.pkl")

# get prediction on test
y_pred = best_model.predict(X_test)

# ---------compute statistics---------
df_results = evaluate_model(y_test, y_pred)
df_results.to_csv(f"best_{MODEL}_{tree_method}_metrics_{OPTIMIZER}_{timestamp}.csv", index=False, float_format="%.4f")

# ---------gest hyper-parameters from best xgb model---------

# Estrai i migliori iperparametri dal RandomizedSearchCV
best_params = random_search.best_params_

# Converti in DataFrame (una riga, colonne = parametri)
df_best_params = pd.DataFrame([best_params])

# Salva in CSV con timestamp
best_params_filename = f"best_{MODEL}_{tree_method}_hyperparameters_{OPTIMIZER}_{timestamp}.csv"
df_best_params.to_csv(best_params_filename, index=False)

In [ ]:
metric_importance = fixed_params["importance_type"] # gain (gbtree or dart) or weight (gblinear)

# Plot feature importance
plt.figure(figsize=(10,6))
plot_importance(best_model, importance_type=metric_importance, max_num_features=20, show_values=True)
plt.title("XGBoost Feature Importance (Gain)")
plt.show()

In [ ]:
# Estrai feature importance come dizionario
importance_dict = best_model.get_booster().get_score(importance_type=metric_importance)

# Converti in DataFrame
df_importance = pd.DataFrame({
    'Feature': list(importance_dict.keys()),
    metric_importance: list(importance_dict.values())
})

# Ordina per importanza decrescente
df_importance = df_importance.sort_values(by=metric_importance, ascending=False)

df_importance.to_csv(f"best_{MODEL}_{tree_method}_feature_importance_{OPTIMIZER}_{timestamp}.csv", index=False, float_format="%.4f")